# 08 — Why the fine-tuning crossover changed: separating the label budget from tuning/LoRA (GPU)

Answers Reviewer 3, point 4, of the second review round (Paper ID 20266676).

The first submission reported that fine-tuning overtakes the frozen pipeline **from 500 texts** (E2b); the revision reports full fine-tuning ahead **from about 250 texts** (regime B). Between the two versions three things changed at once:

1. **the label budget** — E2b let fine-tuning early-stop on the 809 validation labels while the frozen pipeline saw only the *n* training labels; the revision matches the budget (regime A: both see only *n* labels; regime B: both use the validation partition for selection only);
2. **the fine-tuning comparator** — the original recipe (first token, unweighted cross-entropy, constant learning rate 2 × 10⁻⁵ without warm-up, at most 5 epochs, patience 2) was replaced by a tuned recipe (validation search, class-weighted loss, warm-up and decay, 10–40 epochs) and a LoRA comparator;
3. **the design** — nested subsets, 10 instead of 5 seeds, and a new software environment.

The missing cell of this factorial is the **original recipe under the corrected budgets**. This notebook runs it and then decomposes the change:

| Frozen side → / FT side ↓ | FT selects on validation (E2b / regime B) | FT sees only *n* labels (regime A) |
|---|---|---|
| Frozen fits + selects on *n* (regime A) | **E2b design** (first-submission budget) | label-matched |
| Frozen selects on validation (regime B) | selection-matched | — |

× comparator {original recipe (**this notebook**), tuned full FT, tuned LoRA (notebook 04)}. All frozen results come from notebook 03 (`LC_frozen_raw.csv`), all tuned FT from notebook 04 (`ft_runs.csv`), read from the public repository at commit `2c997704beeb8b1c86e1539146864071754ef3bd`.

| Block | Content | Output |
|---|---|---|
| 4 | Fidelity check: the original recipe replays first-submission runs (old independent subsets, seeds 42 and 7, n = 100 and 1,000) | `fidelity_replay.csv` |
| 5 | Original recipe in regime A (n = 100–1,000) and regime B (n = 100–3,510), nested subsets of notebooks 03/04, 10 seeds | `ft_original_runs.csv`, `ft_original_history.csv`, `predictions/` |
| 6 | Δ (frozen − FT) per cell and size with corrected resampled t-test + Holm; crossover summary; paired difference-in-differences for the effect of the budget correction, of tuning and of LoRA | `decomp_cells.csv`, `decomp_crossover_summary.csv`, `decomp_effects.csv`, `decomp_figure.png` |

**Before running on Kaggle**
1. *Add Input*: PRDECT-ID. To resume, add the output of the previous version of this notebook.
2. *Settings*: Accelerator **GPU T4 ×1**, Internet **On** (Hugging Face model + repository clone at the pinned commit).
3. *Save Version → Save & Run All*. Expected ≈ 7–8 h GPU (≈ 40 min per seed + 15 min replay); a session stops cleanly after `TIME_BUDGET_H` and resumes when its output is attached. Block 6 runs on whatever has finished, so it can also be re-run alone.


In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================
import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil, math, copy, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import sklearn, transformers
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from scipy import stats

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
SESSION_T0 = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Configuration

In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))

SEEDS   = [42, 7, 13, 77, 101, 202, 303, 404, 505, 606]
SIZES_B = [100, 250, 500, 750, 1000, 1500, 2500, 3510]     # identical to notebook 04
SIZES_A = [100, 250, 500, 750, 1000]                        # identical to notebook 04
VAL_FRAC, TEST_FRAC = 0.15, 0.20
INNER_VAL_FRAC = 0.20                                      # regime A internal early-stopping split (as notebook 04)

MODEL_NAME, PREFIX, MAX_LEN = "intfloat/multilingual-e5-base", "query: ", 128

# ---- ORIGINAL recipe of the first submission (notebook 01, section 13) -- do not change --------
FT_BATCH_SIZE, FT_LR, FT_WEIGHT_DECAY = 16, 2e-5, 0.01
LC_FT_MAX_EPOCHS, LC_FT_PATIENCE = 5, 2
COMPARATOR = "FT (original recipe)"

RUN_REPLAY = True
REPLAY_SEEDS, REPLAY_SIZES = [42, 7], [100, 1000]
TIME_BUDGET_H = 11.3

REF = "E5 Hybrid (imbalance-aware)"                        # frozen model of the manuscript
N_TOK_TEST, N_TOK_DEV = 1081, 4319

if SMOKE_TEST:
    SEEDS, SIZES_B, SIZES_A = [42, 7], [100, 250], [100]
    REPLAY_SEEDS, REPLAY_SIZES = [42], [100]
    LC_FT_MAX_EPOCHS = 2

# ---- paths ------------------------------------------------------
INPUT_ROOTS = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))
def _find(pattern, roots=INPUT_ROOTS):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits: return hits[0]
    return None

TOKOPEDIA_CSV = _find("PRDECT-ID Dataset.csv") or _find("*PRDECT*.csv") or _find("Tokopedia Review.csv")
assert TOKOPEDIA_CSV, "PRDECT-ID (Tokopedia) CSV not found: add the dataset as input."
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT = os.path.join(WORK, "crossover_decomp")
_prev = [p for r in INPUT_ROOTS for p in glob.glob(os.path.join(r, "**", "crossover_decomp"), recursive=True) if os.path.isdir(p)]
if _prev and not os.path.exists(OUT_ROOT):
    shutil.copytree(_prev[0], OUT_ROOT); print("Resumed from", _prev[0])
for d in [OUT_ROOT, f"{OUT_ROOT}/predictions"]:
    Path(d).mkdir(parents=True, exist_ok=True)
RUNS_CSV, HIST_CSV, REPLAY_CSV = f"{OUT_ROOT}/ft_original_runs.csv", f"{OUT_ROOT}/ft_original_history.csv", f"{OUT_ROOT}/fidelity_replay.csv"

# ---- reference results (immutable commit) -----------------------
REF_COMMIT = "2c997704beeb8b1c86e1539146864071754ef3bd"
REF_URL    = "https://github.com/wagino89/ijies-low-resource-emotion-classification.git"
REPO_DIR   = os.path.join(WORK, "ref_repo")

def repo_root():
    for r in INPUT_ROOTS:
        for h in sorted(glob.glob(os.path.join(r, "**", "results/rerun_frozen/LC_frozen_raw.csv"), recursive=True)):
            root = str(Path(h).parents[2])
            head = subprocess.run(["git", "-C", root, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
            return root, (f"attached input, git {head}" if head else "attached input (no .git: commit not verifiable)")
    if not os.path.exists(f"{REPO_DIR}/.git"):
        cmd = f"git init -q {REPO_DIR} && cd {REPO_DIR} && git remote add origin {REF_URL} && git fetch -q --depth 1 origin {REF_COMMIT} && git checkout -q FETCH_HEAD"
        assert subprocess.run(["bash", "-c", cmd]).returncode == 0, "Clone failed: turn Internet on or attach the repository as input."
    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head == REF_COMMIT, head
    return REPO_DIR, f"git {head}"

REPO, REPO_SOURCE = repo_root()
print("Tokopedia csv:", TOKOPEDIA_CSV, "| output:", OUT_ROOT, "| reference:", REPO_SOURCE, "| SMOKE_TEST =", SMOKE_TEST)

def set_seed(seed):                       # identical to notebook 01
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def append_csv(path, rows):
    new = pd.DataFrame(rows); old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True); df.to_csv(path, index=False); return df

def done_keys(path, cols):
    if not os.path.exists(path): return set()
    return set(map(tuple, pd.read_csv(path)[cols].astype(str).values.tolist()))

def hours_used(): return (time.time() - SESSION_T0) / 3600


## 2. Data, splits, nested subsets (identical to notebooks 03 and 04)

In [ ]:
def load_tokopedia():
    df = pd.read_csv(TOKOPEDIA_CSV)[["Customer Review", "Emotion"]].dropna().reset_index(drop=True)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["Emotion"])
    df["uid"] = [f"tok_{i:05d}" for i in range(len(df))]
    return df, list(le.classes_)

def _two_stage(splitter, y, seed):
    # identical to the original notebook: 65 / 15 / 20 split, stratified, random_state = seed in both stages
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])

LC_ORDER_SALT = 20261023

def lc_order(seed):
    """Nested, approximately stratified ordering of the training partition of one seed.
    The learning-curve subset of size n is the first n entries, so every smaller subset is
    contained in every larger one (nested design). Each class is shuffled and its members are
    spread uniformly over the ranking, so every prefix keeps the class proportions."""
    tr = TOKO_SPLITS[seed][0]; lab = TOKO_DF.label.values[tr]
    rng = np.random.default_rng(LC_ORDER_SALT + seed)
    key = np.empty(len(tr))
    for c in np.unique(lab):
        pos = np.where(lab == c)[0]; m = len(pos)
        key[pos[rng.permutation(m)]] = (np.arange(m) + rng.random(m)) / m
    return np.asarray(tr)[np.argsort(key, kind="stable")]

TOKO_DF, TOKO_CLASSES = load_tokopedia()
K = len(TOKO_CLASSES); Y = TOKO_DF.label.values
TOKO_SPLITS = {s: _two_stage(StratifiedShuffleSplit, Y, s) for s in SEEDS}
LC_ORDER = {s: lc_order(s) for s in SEEDS}

# cross-check with notebook 03 (if its output is attached)
_ord = f"{REPO}/results/rerun_frozen/lc_subset_order.csv"
if _ord:
    o = pd.read_csv(_ord)
    for s in SEEDS:
        ref = o[o.seed == s].sort_values("rank").uid.values
        if len(ref): assert np.array_equal(ref, TOKO_DF.uid.values[LC_ORDER[s]]), f"subset order differs for seed {s}"
    print("Subset order identical to notebook 03:", _ord)

def inner_split(sub, seed, n):
    # regime A: stratified 80/20 split of the n training texts (early stopping only)
    a, b = next(StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC, random_state=seed * 10000 + n)
                .split(np.zeros(len(sub)), Y[sub]))
    return np.asarray(sub)[a], np.asarray(sub)[b]

rows = []
for s in SEEDS:
    for n in SIZES_A:
        a, b = inner_split(LC_ORDER[s][:n], s, n)
        rows += [{"seed": s, "train_size": n, "part": "inner_train", "uid": u} for u in TOKO_DF.uid.values[a]]
        rows += [{"seed": s, "train_size": n, "part": "inner_val", "uid": u} for u in TOKO_DF.uid.values[b]]
pd.DataFrame(rows).to_csv(f"{OUT_ROOT}/regimeA_inner_splits.csv", index=False)
# same internal splits as the tuned comparators of notebook 04
ref_in = pd.read_csv(f"{REPO}/results/rerun_finetune/regimeA_inner_splits.csv")
mine = pd.DataFrame(rows)
for (s_, n_, p_), g in mine.groupby(["seed", "train_size", "part"]):
    r_ = ref_in[(ref_in.seed == s_) & (ref_in.train_size == n_) & (ref_in.part == p_)].uid
    assert set(r_) == set(g.uid), f"regime-A inner split differs from notebook 04 (seed {s_}, n {n_}, {p_})"
print("Regime-A inner splits identical to notebook 04")

# first-submission design (notebook 01): independent stratified subsets, used only by the fidelity replay
def stratified_subset(full_train, n, seed):
    if n >= len(full_train): return np.asarray(full_train)
    a, _ = next(StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
                .split(np.zeros(len(full_train)), Y[full_train]))
    return np.asarray(full_train)[a]

def lc_split_first_submission(seed, n):
    tr, va, te = TOKO_SPLITS[seed]
    return stratified_subset(tr, n, seed + 10000 + n), va, te
print("Tokopedia:", len(TOKO_DF), TOKO_CLASSES, "| seed 42 train/val/test:", [len(x) for x in TOKO_SPLITS[42]])

## 3. Original fine-tuning recipe of the first submission (verbatim from notebook 01)

In [ ]:
# ============================================================
# 3. ORIGINAL FINE-TUNING RECIPE (notebook 01, section 13), parameterised by index sets
#    AutoModelForSequenceClassification (first-token head), AdamW lr 2e-5, weight decay 0.01, batch 16,
#    constant learning rate (no warm-up / scheduler), unweighted cross-entropy, fp32,
#    at most 5 epochs, early stopping with patience 2 on the selection Macro F1, best epoch restored.
# ============================================================
TOKZ = AutoTokenizer.from_pretrained(MODEL_NAME)
TEXTS = TOKO_DF["Customer Review"].values
macro = lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0)

class TxtDS(Dataset):                                      # verbatim from notebook 01
    def __init__(self, texts, y, tok, prefix):
        self.enc = tok([(prefix or "") + str(x) for x in texts], padding="max_length", truncation=True,
                       max_length=MAX_LEN, return_tensors="pt")
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()} | {"labels": self.y[i]}

@torch.no_grad()
def ft_predict(model, tok, texts, prefix):                 # verbatim from notebook 01
    model.eval(); out = []
    for i in range(0, len(texts), 64):
        enc = tok([(prefix or "") + str(x) for x in texts[i:i + 64]], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(model(**enc).logits, -1).cpu().numpy())
    return np.vstack(out)

def fit_original(train_idx, sel_idx, seed, fixed_epochs=None):
    """Original recipe on train_idx. With sel_idx: early stopping on its Macro F1 (as notebook 01).
    With fixed_epochs: train exactly that many epochs and keep the last state (refit of regime A)."""
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=K).to(DEVICE)
    dl = DataLoader(TxtDS(TEXTS[train_idx], Y[train_idx], TOKZ, PREFIX), batch_size=FT_BATCH_SIZE, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
    E = int(fixed_epochs) if fixed_epochs else LC_FT_MAX_EPOCHS
    best, best_state, pat, epochs, best_ep, hist, early = -1.0, None, LC_FT_PATIENCE, 0, 0, [], False
    t0 = time.perf_counter()
    for ep in range(E):
        model.train(); epochs += 1; tl, nb = 0.0, 0
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            opt.zero_grad(); loss = model(**batch).loss; loss.backward(); opt.step(); tl += loss.item(); nb += 1
        rec = {"epoch": ep + 1, "train_loss": tl / nb, "elapsed_s": time.perf_counter() - t0}
        if sel_idx is not None:
            f = macro(Y[sel_idx], ft_predict(model, TOKZ, TEXTS[sel_idx], PREFIX).argmax(1)); rec["sel_macro_f1"] = f
            if f > best + 1e-6:
                best, pat, best_ep = f, LC_FT_PATIENCE, ep + 1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                pat -= 1
                if pat <= 0: hist.append(rec); early = True; break
        hist.append(rec)
    if sel_idx is not None: model.load_state_dict(best_state)
    else: best_ep = E
    info = {"best_sel_macro_f1": best if sel_idx is not None else np.nan, "best_epoch": best_ep, "epochs_run": epochs,
            "max_epochs": E, "stopped_at_cap": (not early) if sel_idx is not None else None,
            "train_time_s": time.perf_counter() - t0}
    return model, info, hist

def free(model):
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def evaluate(model, te):
    P = ft_predict(model, TOKZ, TEXTS[te], PREFIX); yp = P.argmax(1); yt = Y[te]
    return P, {"macro_f1": macro(yt, yp), "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
               "accuracy": accuracy_score(yt, yp), "n_pred_classes": int(len(np.unique(yp)))}


## 4. Fidelity check: replay of first-submission runs

In [ ]:
# ============================================================
# 4. FIDELITY CHECK — the original recipe replays first-submission runs (old independent subsets, E2b budget)
#    Exact equality is not expected (different GPU kernels and library versions); the scores should be close.
# ============================================================
OLD_FT = pd.read_csv(f"{REPO}/results/main/learning_curve_E5_finetuned.csv")
if RUN_REPLAY:
    done = done_keys(REPLAY_CSV, ["seed", "train_size"])
    for seed in REPLAY_SEEDS:
        for n in REPLAY_SIZES:
            if (str(seed), str(n)) in done: continue
            tr, va, te = lc_split_first_submission(seed, n)
            model, info, hist = fit_original(tr, va, seed); P, sc = evaluate(model, te); free(model)
            old = OLD_FT[(OLD_FT.seed == seed) & (OLD_FT.train_size == n)].iloc[0]
            append_csv(REPLAY_CSV, [{"seed": seed, "train_size": n, "macro_f1_replay": sc["macro_f1"], "macro_f1_first_submission": old.macro_f1,
                                     "diff": sc["macro_f1"] - old.macro_f1, "epochs_replay": info["epochs_run"], "epochs_first_submission": old.epochs_run,
                                     "best_val_replay": info["best_sel_macro_f1"], "best_val_first_submission": old.best_val_macro_f1,
                                     "pred_classes_replay": sc["n_pred_classes"], "pred_classes_first_submission": old.get("n_pred_classes", np.nan),
                                     "train_time_s": info["train_time_s"]}])
            print(f"replay seed={seed} n={n}: {sc['macro_f1']:.4f} vs first submission {old.macro_f1:.4f} ({info['train_time_s']:.0f}s)")
if os.path.exists(REPLAY_CSV): print(pd.read_csv(REPLAY_CSV).round(4).to_string(index=False))


## 5. Original recipe under regimes A and B (10 seeds)

In [ ]:
# ============================================================
# 5. ORIGINAL RECIPE UNDER REGIMES A AND B (seed-major, resumable, stops before the time budget)
#    A_label_matched     : early stopping on the 80/20 inner split of the n texts (same split as notebook 04),
#                          then refit on all n texts for the selected number of epochs
#    B_selection_matched : train on the n texts, early stopping on the fixed 809-text validation partition
#                          (= the fine-tuning side of the first-submission E2b, now on nested subsets and 10 seeds)
# ============================================================
_t = OLD_FT.groupby("train_size").train_time_s.mean()
T_EST = lambda n: float(np.interp(n, _t.index.values, _t.values))           # seconds, from the first submission (T4)

def record(regime, seed, n, model, info, hist, extra=None):
    te = TOKO_SPLITS[seed][2]; P, sc = evaluate(model, te)
    row = {"regime": regime, "comparator": COMPARATOR, "seed": seed, "train_size": n, **sc, **info, **(extra or {})}
    tag = f"{regime}_original_seed{seed}_n{n}"
    np.savez_compressed(f"{OUT_ROOT}/predictions/{tag}.npz", proba=P.astype(np.float16), test_uid=TOKO_DF.uid.values[te])
    append_csv(RUNS_CSV, [row]); append_csv(HIST_CSV, [{"run": f"{tag}|{h_tag}", **h} for h_tag, hs in hist for h in hs])
    print(f"{regime:20s} seed={seed:<3} n={n:<5} macroF1={row['macro_f1']:.4f} best_ep={info['best_epoch']}/{info['epochs_run']} "
          f"({info['train_time_s']:.0f}s) | {hours_used():.2f} h used")

print(f"Projected: {sum(T_EST(n) for n in SIZES_B) / 60 + sum(1.8 * T_EST(n) for n in SIZES_A) / 60:.0f} min per seed")
stop = False
for seed in SEEDS:
    if stop: break
    tr, va, te = TOKO_SPLITS[seed]; done = done_keys(RUNS_CSV, ["regime", "seed", "train_size"])
    jobs = [("A_label_matched", n) for n in SIZES_A] + [("B_selection_matched", n) for n in SIZES_B]
    for regime, n in jobs:
        if (regime, str(seed), str(n)) in done: continue
        est = (1.8 if regime.startswith("A") else 1.0) * T_EST(n) / 3600
        if hours_used() + est > TIME_BUDGET_H:
            print(f"Time budget reached ({hours_used():.2f} h). Re-run with this version's output attached to continue."); stop = True; break
        sub = LC_ORDER[seed][:n]
        if regime.startswith("A"):
            itr, iva = inner_split(sub, seed, n)
            m1, info1, h1 = fit_original(itr, iva, seed); free(m1)
            model, info, h2 = fit_original(sub, None, seed, fixed_epochs=info1["best_epoch"])
            extra = {"inner_best_epoch": info1["best_epoch"], "inner_epochs_run": info1["epochs_run"],
                     "inner_stopped_at_cap": info1["stopped_at_cap"], "inner_val_macro_f1": info1["best_sel_macro_f1"],
                     "selection_time_s": info1["train_time_s"]}
            record(regime, seed, n, model, info, [("inner", h1), ("refit", h2)], extra)
        else:
            model, info, h = fit_original(sub, va, seed)
            record(regime, seed, n, model, info, [("main", h)])
        free(model)
    if not stop: print(f"=== seed {seed} complete | {hours_used():.2f} h used ===")


## 6. Decomposition of the crossover change

In [ ]:
# ============================================================
# 6. DECOMPOSITION OF THE CROSSOVER CHANGE (CPU, seconds; runs on whatever has finished)
#    Δ = frozen (E5 Hybrid, imbalance-aware) − fine-tuned, paired by seed. Corrected resampled t-test
#    (Nadeau–Bengio; primary ratio 1081/4319, conservative 1081/n), Holm across sizes within each cell.
# ============================================================
def corrected_ttest(d, ratio):                      # identical to scripts/analyze_rerun.py
    d = np.asarray(d, float); J = len(d); m = d.mean()
    if J < 2: return m, np.nan, np.nan, np.nan
    se = np.sqrt((1.0 / J + ratio) * d.var(ddof=1))
    if se == 0: return m, m, m, 1.0
    t = m / se; h = stats.t.ppf(0.975, J - 1) * se
    return m, m - h, m + h, 2 * stats.t.sf(abs(t), J - 1)

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    q = p[ok]; o = np.argsort(q); adj = np.empty_like(q); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (len(q) - r) * q[i]); adj[i] = min(1.0, run)
    out[ok] = adj; return out

LF   = pd.read_csv(f"{REPO}/results/rerun_frozen/LC_frozen_raw.csv")
FT_T = pd.read_csv(f"{REPO}/results/rerun_finetune/ft_runs.csv")
FT_O = pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=["regime", "seed", "train_size", "macro_f1"])

fz = lambda reg: LF[(LF.model == REF) & (LF.regime == reg)]
ft_t = lambda reg, comp: FT_T[(FT_T.regime == reg) & (FT_T.comparator == comp)]
ft_o = lambda reg: FT_O[FT_O.regime == reg]
CELLS = [  # (cell id, budget, comparator, frozen runs, FT runs)
    ("E2b|original", "E2b budget (first submission): frozen on n, FT selects on validation", "original recipe", fz("A_label_matched"), ft_o("B_selection_matched")),
    ("E2b|tuned",    "E2b budget (first submission): frozen on n, FT selects on validation", "tuned full FT",   fz("A_label_matched"), ft_t("B_selection_matched", "FT (tuned)")),
    ("E2b|lora",     "E2b budget (first submission): frozen on n, FT selects on validation", "tuned LoRA",      fz("A_label_matched"), ft_t("B_selection_matched", "LoRA (tuned)")),
    ("A|original",   "Regime A (label-matched)",     "original recipe", fz("A_label_matched"), ft_o("A_label_matched")),
    ("A|tuned",      "Regime A (label-matched)",     "tuned full FT",   fz("A_label_matched"), ft_t("A_label_matched", "FT (tuned)")),
    ("B|original",   "Regime B (selection-matched)", "original recipe", fz("B_selection_matched"), ft_o("B_selection_matched")),
    ("B|tuned",      "Regime B (selection-matched)", "tuned full FT",   fz("B_selection_matched"), ft_t("B_selection_matched", "FT (tuned)")),
    ("B|lora",       "Regime B (selection-matched)", "tuned LoRA",      fz("B_selection_matched"), ft_t("B_selection_matched", "LoRA (tuned)")),
]

DELTA = {}           # cell -> DataFrame(seed x size) of frozen − FT
rows = []
for cid, budget, comp, F_, T_ in CELLS:
    block = []
    for n in sorted(set(F_.train_size) & set(T_.train_size)):
        a = F_[F_.train_size == n].set_index("seed").macro_f1; b = T_[T_.train_size == n].set_index("seed").macro_f1
        j = a.index.intersection(b.index)
        if len(j) < 2: continue
        d = (a[j] - b[j]); DELTA.setdefault(cid, {})[n] = d
        m, lo, hi, p = corrected_ttest(d.values, N_TOK_TEST / N_TOK_DEV); _, lo_c, hi_c, p_c = corrected_ttest(d.values, N_TOK_TEST / n)
        block.append(dict(cell=cid, budget=budget, comparator=comp, train_size=n, n_seeds=len(j), frozen_mean=a[j].mean(), ft_mean=b[j].mean(),
                          delta=m, ci95_low=lo, ci95_high=hi, p_corrected=p, p_corrected_conservative=p_c,
                          ci95_low_conservative=lo_c, ci95_high_conservative=hi_c, frozen_wins=int((d > 0).sum())))
    if block:
        b_ = pd.DataFrame(block); b_["p_holm"] = holm(b_.p_corrected); b_["p_holm_conservative"] = holm(b_.p_corrected_conservative); rows.append(b_)

# first-submission E2b (5 seeds, independent subsets, earlier environment) as published, same statistics
OLD_FZ = pd.read_csv(f"{REPO}/results/main/learning_curve_ME_XBAL_matched_budget.csv")
blk = []
for n in sorted(set(OLD_FZ.train_size) & set(OLD_FT.train_size)):
    a = OLD_FZ[OLD_FZ.train_size == n].set_index("seed").macro_f1; b = OLD_FT[OLD_FT.train_size == n].set_index("seed").macro_f1
    j = a.index.intersection(b.index); d = a[j] - b[j]
    m, lo, hi, p = corrected_ttest(d.values, N_TOK_TEST / N_TOK_DEV); _, lo_c, hi_c, p_c = corrected_ttest(d.values, N_TOK_TEST / n)
    blk.append(dict(cell="first submission E2b", budget="E2b budget, first submission (5 seeds, independent subsets)", comparator="original recipe",
                    train_size=n, n_seeds=len(j), frozen_mean=a[j].mean(), ft_mean=b[j].mean(), delta=m, ci95_low=lo, ci95_high=hi,
                    p_corrected=p, p_corrected_conservative=p_c, ci95_low_conservative=lo_c, ci95_high_conservative=hi_c, frozen_wins=int((d > 0).sum())))
b_ = pd.DataFrame(blk); b_["p_holm"] = holm(b_.p_corrected); b_["p_holm_conservative"] = holm(b_.p_corrected_conservative)
TAB = pd.concat([b_] + rows, ignore_index=True); TAB.to_csv(f"{OUT_ROOT}/decomp_cells.csv", index=False)

# crossover summary per cell
summ = []
for cid, g in TAB.groupby("cell", sort=False):
    g = g.sort_values("train_size")
    for col in ["p_holm", "p_holm_conservative"]:
        fzb = g[(g[col] < 0.05) & (g.delta > 0)].train_size; ftb = g[(g[col] < 0.05) & (g.delta < 0)].train_size
        first_ft_ahead = g[g.delta < 0].train_size
        summ.append(dict(cell=cid, budget=g.budget.iloc[0], comparator=g.comparator.iloc[0], test=col, seeds=int(g.n_seeds.max()),
                         frozen_significantly_better_at=",".join(map(str, fzb)) or "-",
                         ft_significantly_better_at=",".join(map(str, ftb)) or "-",
                         ft_numerically_ahead_from=int(first_ft_ahead.min()) if len(first_ft_ahead) else None,
                         ft_significantly_ahead_from=int(ftb.min()) if len(ftb) else None,
                         delta_at_100=float(g[g.train_size == 100].delta.iloc[0]) if (g.train_size == 100).any() else np.nan))
SUMM = pd.DataFrame(summ); SUMM.to_csv(f"{OUT_ROOT}/decomp_crossover_summary.csv", index=False)

# paired difference-in-differences: how much each change moves Δ (same seeds, same nested subsets)
EFFECTS = [  # (label, factor, cell after, cell before)
    ("Budget correction, FT loses the validation labels (E2b -> A), original recipe", "label budget", "A|original", "E2b|original"),
    ("Budget correction, frozen gains validation selection (E2b -> B), original recipe", "label budget", "B|original", "E2b|original"),
    ("Tuning of full FT under the E2b budget", "tuning", "E2b|tuned", "E2b|original"),
    ("Tuning of full FT under regime A", "tuning", "A|tuned", "A|original"),
    ("Tuning of full FT under regime B", "tuning", "B|tuned", "B|original"),
    ("LoRA instead of original recipe under regime B", "tuning + LoRA", "B|lora", "B|original"),
    ("LoRA instead of tuned full FT under regime B", "LoRA", "B|lora", "B|tuned"),
    ("Budget correction with tuned FT (E2b -> A)", "label budget", "A|tuned", "E2b|tuned"),
    ("Budget correction with tuned FT (E2b -> B)", "label budget", "B|tuned", "E2b|tuned"),
]
eff = []
for lab, factor, after, before in EFFECTS:
    if after not in DELTA or before not in DELTA: continue
    block = []
    for n in sorted(set(DELTA[after]) & set(DELTA[before])):
        x, y = DELTA[after][n], DELTA[before][n]; j = x.index.intersection(y.index)
        if len(j) < 2: continue
        dd = (x[j] - y[j]).values; m, lo, hi, p = corrected_ttest(dd, N_TOK_TEST / N_TOK_DEV)
        block.append(dict(effect=lab, factor=factor, cell_after=after, cell_before=before, train_size=n, n_seeds=len(j),
                          delta_before=y[j].mean(), delta_after=x[j].mean(), change_in_delta=m, ci95_low=lo, ci95_high=hi, p_corrected=p))
    if block:
        b_ = pd.DataFrame(block); b_["p_holm"] = holm(b_.p_corrected); eff.append(b_)
EFF = pd.concat(eff, ignore_index=True) if eff else pd.DataFrame(); EFF.to_csv(f"{OUT_ROOT}/decomp_effects.csv", index=False)

pd.set_option("display.width", 250)
print("Δ (frozen − FT) by cell and size:")
print(TAB.pivot_table(index="cell", columns="train_size", values="delta", sort=False).round(4).to_string())
print("\nCrossover summary (Holm, primary ratio):")
print(SUMM[SUMM.test == "p_holm"].drop(columns=["test", "budget"]).to_string(index=False))
if len(EFF):
    print("\nChange in Δ (negative = the change favours fine-tuning):")
    print(EFF.pivot_table(index="effect", columns="train_size", values="change_in_delta", sort=False).round(4).to_string())

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharey=True)
    for ax, (title, ids) in zip(axes, [("E2b budget (first submission)", ["first submission E2b", "E2b|original", "E2b|tuned", "E2b|lora"]),
                                       ("Regime A: label-matched", ["A|original", "A|tuned"]),
                                       ("Regime B: selection-matched", ["B|original", "B|tuned", "B|lora"])]):
        for cid in ids:
            g = TAB[TAB.cell == cid].sort_values("train_size")
            if len(g): ax.errorbar(g.train_size, g.delta, yerr=[g.delta - g.ci95_low, g.ci95_high - g.delta], marker="o", capsize=3,
                                   label=f"{cid} ({int(g.n_seeds.max())} seeds)")
        ax.axhline(0, color="k", lw=.8); ax.set_xscale("log"); ax.set_title(title); ax.set_xlabel("Training texts (log scale)"); ax.grid(alpha=.25)
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Δ Macro F1 (frozen − fine-tuned)"); fig.tight_layout(); fig.savefig(f"{OUT_ROOT}/decomp_figure.png", dpi=300)
except Exception as e:
    print("figure skipped:", e)


## 7. Environment record and archive

In [ ]:
# ============================================================
# 7. ENVIRONMENT + ZIP
# ============================================================
if os.path.exists(RUNS_CSV):
    r = pd.read_csv(RUNS_CSV)
    print(r.groupby(["regime"]).agg(runs=("seed", "size"), seeds=("seed", "nunique"), stopped_at_cap=("stopped_at_cap", lambda x: pd.to_numeric(x, errors="coerce").mean()),
                                    mean_best_epoch=("best_epoch", "mean"), mean_pred_classes=("n_pred_classes", "mean")).round(2))
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "amp_fp16": False,
       "model": MODEL_NAME, "prefix": PREFIX, "max_len": MAX_LEN,
       "recipe": {"head": "AutoModelForSequenceClassification (first token)", "lr": FT_LR, "weight_decay": FT_WEIGHT_DECAY,
                  "batch": FT_BATCH_SIZE, "scheduler": None, "warmup": None, "loss": "unweighted cross-entropy",
                  "max_epochs": LC_FT_MAX_EPOCHS, "patience": LC_FT_PATIENCE, "precision": "fp32"},
       "seeds": SEEDS, "sizes_A": SIZES_A, "sizes_B": SIZES_B, "inner_val_frac": INNER_VAL_FRAC,
       "reference": REPO_SOURCE, "reference_commit": REF_COMMIT, "smoke_test": SMOKE_TEST, "session_hours": round(hours_used(), 2)}
json.dump(env, open(f"{OUT_ROOT}/environment_gpu.json", "w"), indent=2)
try:
    open(f"{OUT_ROOT}/pip_freeze_gpu.txt", "w").write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
except Exception as e: print("pip freeze failed:", e)
shutil.make_archive(f"{WORK}/crossover_decomp_outputs", "zip", OUT_ROOT)
print("Archive:", f"{WORK}/crossover_decomp_outputs.zip")
